# Week 2: K-Means Clustering on the HR Diagram

**Goal:** Use unsupervised machine learning to automatically group stars into clusters on the HR diagram — without any predefined labels.

**Why K-Means here?**  
Astronomers know that stars fall into distinct evolutionary groups (main sequence, red giants, white dwarfs). K-Means should recover these groups purely from the data's geometry in color-magnitude space.

**Prerequisite:** Run all cells in `week1_gaia_exploration.ipynb` first, or just re-run the data setup cells below (they duplicate week1's fetch + clean + features steps).

---

**Notebook flow:**  
`Setup → Load & Clean Data → Elbow Method → Fit K-Means → Visualize Clusters → Interpret`

## Setup

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from src.data_fetch import fetch_gaia_sample
from src.data_clean import clean_sample, add_features
from src import visualize

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

## Step 1: Load and Clean Data

Same pipeline as week1. We need `bp_rp` and `absolute_mag` as our two clustering features.

In [ ]:
df = fetch_gaia_sample(top_n=10000)
clean = clean_sample(df)
clean = add_features(clean)
print(f"Ready to cluster: {len(clean):,} stars")

## Step 2: Scale Features

K-Means uses Euclidean distance. If one feature has a much larger range than another, it dominates the distance calculation and the other feature is effectively ignored.

`StandardScaler` transforms each feature to mean=0, std=1 so both `bp_rp` and `absolute_mag` contribute equally to cluster assignment.

In [ ]:
features = clean[["bp_rp", "absolute_mag"]]

# Fit the scaler on the feature matrix and transform it
scaler = StandardScaler()
X = scaler.fit_transform(features)

print(f"Feature matrix shape: {X.shape}  (rows=stars, cols=features)")

## Step 3: Elbow Method — Choose the Right k

K-Means requires you to specify the number of clusters `k` in advance. The **elbow method** helps pick a good value:

- For each candidate `k`, fit K-Means and record **inertia** — the sum of squared distances from each point to its assigned cluster center.
- Inertia always decreases as `k` increases (more clusters = smaller groups = closer to centroids).
- Look for the **elbow**: the point where adding another cluster stops giving a significant improvement.

For the HR diagram, we expect k=4 to work well (main sequence, red giants, subgiants, white dwarfs) — but let the elbow plot guide you.

In [ ]:
inertias = []
k_range = range(2, 10)

for k in k_range:
    # n_init=10: run K-Means 10 times with different random starts, keep the best
    # This avoids getting stuck in a bad local minimum
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), inertias, marker="o", color="steelblue")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia (sum of squared distances)")
plt.title("Elbow Method — Pick k where the curve bends")
plt.tight_layout()
plt.show()

## Step 4: Fit K-Means

We use k=4, targeting the four main stellar populations visible on the HR diagram:  
main sequence, red giants, subgiants, and white dwarfs.

Adjust `N_CLUSTERS` if your elbow plot suggests a different value.

In [ ]:
N_CLUSTERS = 4

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
clean["cluster"] = kmeans.fit_predict(X)

print("Stars per cluster:")
print(clean["cluster"].value_counts().sort_index())

## Step 5: Visualize Clusters on the HR Diagram

In [ ]:
visualize.plot_hr_clusters(clean, n_clusters=N_CLUSTERS)

## Step 6: Interpret the Clusters

Look at the mean properties of each cluster to identify which stellar population it corresponds to.

**Reading the table:**
- Low `bp_rp` → blue/hot star  
- High `bp_rp` → red/cool star  
- Low `absolute_mag` → intrinsically bright  
- High `absolute_mag` → intrinsically dim  

**Expected populations:**
| Population | bp_rp | absolute_mag | Notes |
|------------|-------|--------------|-------|
| Main sequence (hot) | 0.5–1.5 | 2–5 | Upper main sequence |
| Main sequence (cool) | 1.5–3.0 | 5–10 | Lower main sequence |
| Red giants | 1.5–3.0 | -1 to 2 | Bright + red |
| White dwarfs | 0.0–0.5 | 10–15 | Dim + hot/blue |

In [ ]:
summary = (
    clean.groupby("cluster")[["bp_rp", "absolute_mag", "distance_pc"]]
    .mean()
    .round(2)
)
summary.columns = ["Mean BP-RP (color)", "Mean Abs Mag (brightness)", "Mean Distance (pc)"]
print(summary)
print()
print("Low bp_rp = hot/blue  |  High bp_rp = cool/red")
print("Low abs_mag = bright  |  High abs_mag = dim")

---

## Summary

| Step | What we did | Why |
|------|-------------|-----|
| Scale | StandardScaler on bp_rp + absolute_mag | Equal feature contribution to distance |
| Elbow method | Inertia vs. k plot | Pick k without guessing |
| K-Means (k=4) | Fit and assign cluster labels | Group stars by position in color-magnitude space |
| Visualize | Color-coded HR diagram | See if clusters match known stellar populations |
| Interpret | Mean properties per cluster | Connect cluster numbers to astrophysical meaning |

**Key takeaway:** Unsupervised clustering recovers physically meaningful stellar groups purely from photometric measurements — no labels required.